# Demo of Tomographic Sigma8 bias Metric¶

**Notebook overview (added for clarity).**

This notebook uses `rubin_sim.maf` (the LSST Metric Analysis Framework, MAF) to evaluate how a given
observing strategy / cadence (an OpSim run) affects DESC (Dark Energy Science Collaboration) cosmology
science, by translating survey non-uniformity (depth fluctuations across the sky and over time) into
biases on cosmological quantities. It contains three independent demos, each built the same way:

1. A per-pixel (Healpix) 'parent' Metric computes some quantity from the visits in each sky pixel
   (e.g. coadded depth, or depth converted into a galaxy density/redshift perturbation).
2. A 'summary' Metric then reduces the resulting Healpix map (or set of maps) down to one number
   (or a few numbers) that quantifies the cosmological impact (a bias on sigma8, a loss of Figure of
   Merit, a bias on mean redshift, etc.), usually by comparing the observed non-uniformity map to a
   theoretical/fiducial DESC model (angular power spectra, dn/dz, etc.).
3. This is repeated for each 'year' of the survey (year 1 to year 10) to see how the metric evolves
   as the survey accumulates depth.

The three demos are:
- **Demo 1 (this section): Tomographic sigma8 bias.** How much does non-uniform depth bias the
  inferred amplitude of matter clustering (sigma8) in each of 5 tomographic (redshift) bins?
- **Demo 2: AreaAtRisk / FoM ratio.** What fraction of the ideal cosmological Figure of Merit is lost
  because some sky area doesn't reach the required depth/uniformity?
- **Demo 3: Meanz bias.** How much does non-uniform depth bias the mean redshift of each tomographic
  bin (important for weak lensing calibration)?

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import healpy as hp
import pandas as pd

import rubin_sim
import rubin_sim.maf as maf


print(rubin_sim.__version__)
from os.path import splitext, basename
from rubin_scheduler.scheduler.utils import (
    SkyAreaGenerator,
)  # generates reference sky-footprint maps (e.g. the WFD 'lowdust' region) used to build the Healpix slicer below

from rubin_sim.data import (
    get_baseline,
)  # path to the baseline OpSim database, i.e. the simulated LSST cadence/strategy to analyze

In [ ]:
from rubin_sim.maf.metrics.uniformity_metrics import NestedLinearMultibandModelMetric

# NestedLinearMultibandModelMetric: a per-Healpix-pixel ('parent') metric. For each pixel it first computes the
# per-band coadded depth (m5) from the visits that fall in it (internally via ExgalM5WithCuts-like logic), then
# applies a LINEAR model, dlogN/dm5 (how the log galaxy density changes with depth, per tomographic bin and per
# band), to turn that depth fluctuation into a predicted galaxy DENSITY fluctuation for each of the 5 DESC
# tomographic (redshift) bins. Output: one density-fluctuation value per bin, per pixel.
from rubin_sim.maf.metrics.cosmology_summary_metrics import TomographicClusteringSigma8biasMetric
# TomographicClusteringSigma8biasMetric: a 'summary' metric. It takes the 5 density-fluctuation Healpix maps
# produced above, computes their angular power spectra (Cell) up to a bin-dependent lmax, compares that spurious
# power to the fiducial DESC theory power spectra (from CCL, stored in the tomography model), and returns the
# resulting BIAS on sigma8^2 (in units of sigma, or converted to sigma8) that the non-uniform depth would induce
# in a clustering analysis.

In [ ]:
from rubin_sim.maf.metrics.tomography_models import DENSITY_TOMOGRAPHY_MODEL
# this contains the current model.
# the first set of keys are the years (year1, ..., year10) since this would change typical depth and galaxy catalog cuts.
# in what follows we have 5 tomographic bins.
# the second nested dictionary has the following:
# sigma8square_model is the fiducial sigma8^2 value used in CCL for the theory predictions
# poly1d_coefs_loglog is a polynomial (5th degree) describing the angular power spectra (in log log space) in the 5 tomographic bins considered, thus has shape (5, 6)
# lmax contains the lmax limits to sum the Cells over when calculating sigma8 for each tomographic bin. thus is it of shape (5, )
# dlogN_dm5 contains the derivatives of logN wrt m5 calculated in Qianjun & Jeff's simulations. It is an array of 5 dictionaries (5 = the tomographic bins)
# each dictionary must have keys that are the lsst bands. If some are missing they are ignored in the linear model.
# they are the ones which will be fed to LinearMultibandModelMetric. Everything else above is going into the modeling
# The notebook I used to make this dictionary is https://github.com/ixkael/ObsStrat/blob/meanz_uniformity_maf/code/meanz_uniformity/romanrubinmock_for_sigma8tomography.ipynb

In [ ]:
%pinfo maf.NestedLinearMultibandModelMetric

## 1. Configuration

In [ ]:
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
# Baseline Survey
baseline_file = get_baseline()
# opsdb = maf.db.OpsimDatabase(baseline_file)
# opsdb =  maf.db.add_run_to_database(baseline_file)
run_name = os.path.split(baseline_file)[-1].replace(".db", "")

print(run_name)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="01_maf_testSNIa_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

##  Define the slicer and Metrics, Metrics Bundles and Metrics Group

In [ ]:
# to view the signature of the Metrics class
%pinfo SkyAreaGenerator

In [ ]:
# to view the code of the metrics
# %psource SkyAreaGenerator

In [ ]:
%pinfo  NestedLinearMultibandModelMetric

In [ ]:
# to view the code of the metrics
# %psource NestedLinearMultibandModelMetric

In [ ]:
# a simple wrapper around the metrics, to store the results, but not critically needed
def extract_sigma8_tomography_metric(
    opsim_fname,
    run_fname,
    years,
    percentage_uncorrected,
    density_tomography_model,
    lmin=10,
    mag_range_tolerated=1.0,
    n_filters=6,
    extinction_cut=0.2,  # sky cuts
    nside=32,
    convert_to_sigma8=True,
):
    """Compute the DESC tomographic sigma8-bias metric for the given OpSim run and years.

    For each requested survey year, this builds a Healpix map of the depth reached by the survey
    (masked to the WFD 'lowdust' footprint), converts it into a per-tomographic-bin galaxy density
    fluctuation map via ``NestedLinearMultibandModelMetric`` and the year's dlogN/dm5 linear model,
    then reduces those maps to a single number - the resulting bias on sigma8 (or sigma8^2) - via
    ``TomographicClusteringSigma8biasMetric``.

    Parameters
    ----------
    opsim_fname : str
        Path to the OpSim SQLite database (the simulated cadence) to run the metric on.
    run_fname : str
        Name of the run, used to label the MAF MetricBundle (not the file path).
    years : iterable of int
        Survey years to evaluate (e.g. ``range(1, 11)`` for years 1 to 10); each year sets the
        ``night <=`` cut used to select the visits accumulated up to that point.
    percentage_uncorrected : float
        Fraction of the depth-driven spurious power assumed NOT removed by calibration/masking
        (passed to ``TomographicClusteringSigma8biasMetric`` as ``power_multiplier``).
    density_tomography_model : dict
        DESC tomography model (e.g. ``DENSITY_TOMOGRAPHY_MODEL``), keyed by ``year1``...``year10``,
        giving per year the fiducial sigma8^2, the Cell polynomial fits, the per-bin lmax, and the
        dlogN/dm5 coefficients.
    lmin : int, optional
        Smallest multipole included when summing the angular power spectra (default 10).
    mag_range_tolerated : float, optional
        Currently unused inside the function body; kept for interface compatibility (default 1.0).
    n_filters : int, optional
        Minimum number of filters required per pixel (depth-map quality cut, default 6).
    extinction_cut : float, optional
        Maximum Galactic dust extinction allowed per pixel (sky cut, default 0.2).
    nside : int, optional
        Healpix resolution used for the slicer and for the density-fluctuation maps (default 32).
    convert_to_sigma8 : bool, optional
        If True, report the bias on sigma8; if False, report the bias on sigma8^2 (default True).

    Returns
    -------
    results_sigma8_squared_bias : numpy.ndarray
        Array of shape ``(len(years),)`` with the sigma8 (or sigma8^2) bias for each requested year.
    all_depth_map_bundles : list of maf.MetricBundle
        The MetricBundle produced for each year, in case the underlying maps/results are needed.
    """

    surveyAreas = SkyAreaGenerator(nside=nside)
    map_footprints, map_labels = surveyAreas.return_maps()

    # Healpix slicer
    slicer = maf.HealpixSubsetSlicer(
        nside=nside,
        hpid=np.where(map_labels == "lowdust")[0],  # hpid=np.arange(hp.nside2npix(nside)),
        use_cache=False,
    )

    # prepare empty arrays to fill in the results
    n_bins = 5  # set to 5
    results_spuriousdensitypower = np.zeros((len(years), n_bins))
    results_sigma8_squared_bias = np.zeros((len(years),))

    # loop over years

    all_depth_map_bundles = []
    for iy, year in enumerate(years):
        print("year", year)

        # constraints
        days = year * 365.25
        constraint_str = (
            'scheduler_note not like "DD%" and night <= XX and scheduler_note not like "twilight_near_sun" '
        )
        constraint_str = constraint_str.replace("XX", "%d" % days)

        # leave empty if not specified
        mean_depth = {}
        min_depth_cut = {}
        max_depth_cut = {}

        ##############################
        # now converts depth fluctuations to density fluctuations
        ##############################
        # metric (per-pixel): compute per-band depth in this pixel/year, apply the year's dlogN/dm5 linear
        # response model, and output a predicted density fluctuation for each of the 5 tomographic bins.
        metric = NestedLinearMultibandModelMetric(
            density_tomography_model["year" + str(year)][
                "dlogN_dm5"
            ],  # dlogN/dm5 coefficients per band, per tomographic bin, for this year
            extinction_cut=extinction_cut,  # exclude pixels with galactic extinction above this threshold (sky/dust cut)
            n_filters=n_filters,  # cuts going into ExgalM5WithCuts
            mean_depth=mean_depth,
            min_depth_cut=min_depth_cut,
            max_depth_cut=max_depth_cut,
        )
        # summary metric measures total power via angular power spectra of healpix map (thus needs nside)
        # _but_ has a bin-dependent lmax to consider same scales to consider the same scales as a fct of redshift
        # summary metric (map -> single number): consumes the 5 density-fluctuation maps produced by 'metric'
        # above, computes their Cell power spectra, and converts the excess/spurious power into a bias on
        # sigma8^2 (or sigma8 if convert_to_sigma8=True), relative to the fiducial model in density_tomography_model.
        summary_metrics = [
            TomographicClusteringSigma8biasMetric(
                density_tomography_model[
                    "year" + str(year)
                ],  # fiducial DESC model (sigma8^2, Cell polynomial fits, lmax) for this year
                convert_to_sigma8=convert_to_sigma8,  # report bias on sigma8 (True) or sigma8^2 (False)
                power_multiplier=percentage_uncorrected,  # fraction of the systematic depth-driven power assumed NOT removed by calibration/masking
                lmin=lmin,  # smallest multipole included in the Cell sum (large scales are usually noisy/uncertain, so excluded below lmin)
            ),
        ]
        # then standard way of packing MetricBundles into a MetricBundleGroup
        depth_map_bundles = [
            maf.MetricBundle(
                metric=metric,
                slicer=slicer,
                constraint=constraint_str,
                run_name=run_name,
                summary_metrics=summary_metrics,
            )
        ]
        bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
        bgroup = maf.MetricBundleGroup(bd, opsim_fname)
        bgroup.run_all()

        # compute bias
        # should probably also return fsky
        results_sigma8_squared_bias[iy] = depth_map_bundles[0].summary_values[
            "TomographicClusteringSigma8bias"
        ]
        all_depth_map_bundles.append(depth_map_bundles[0])

    return results_sigma8_squared_bias, all_depth_map_bundles

In [ ]:
# baseline_file = get_baseline()
sim_list = [baseline_file]
name_list = [splitext(basename(sim))[0] for sim in sim_list]

years = range(1, 11)  # [1, 2, 4, 7, 10]#
percentage_uncorrected = 0.1

# large mag_range_tolerated and no min depth in order to make comparison fair between strategies etc
results_fsky = {}
results_sigma8_squared_bias = {}
for opsim_fname, run_name in zip(sim_list, name_list):
    print("run_name:", run_name)
    results_sigma8_squared_bias[run_name], _ = extract_sigma8_tomography_metric(
        opsim_fname,
        run_name,
        years,
        percentage_uncorrected,
        DENSITY_TOMOGRAPHY_MODEL,
        nside=64,
        lmin=10,
        n_filters=6,
        extinction_cut=0.2,
        mag_range_tolerated=2.0,
        convert_to_sigma8=True,
    )

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

colors = ["orange", "blue", "black", "red"]
for i, run_name in enumerate(results_sigma8_squared_bias.keys()):
    axs[0].plot(years, results_sigma8_squared_bias[run_name], label=run_name, marker="o", color=colors[i])
axs[0].legend()

results_sigma8_squared_bias.keys()
axs[1].set_xlabel("Years")
axs[0].set_ylabel("Bias in sigma8 in units of sigmas")

run_name_ = list(results_sigma8_squared_bias.keys())[0]
for i, run_name in enumerate(results_sigma8_squared_bias.keys()):
    axs[1].plot(years, np.array(years) * 0, ls="--", c="orange")
    if run_name != run_name_:
        axs[1].plot(
            years,
            results_sigma8_squared_bias[run_name] - results_sigma8_squared_bias[run_name_],
            label=run_name,
            marker="o",
            color=colors[i],
        )
axs[1].set_ylabel("Top panel minus " + run_name_)

In [ ]:
#!pip install george

# Demo of AreaAtRisk metric

This demo asks: given the depth reached in each pixel/year, what fraction of the ideal cosmological
Figure of Merit (FoM) is retained, once pixels that don't reach the required depth (or are too
non-uniform) are down-weighted or excluded ('at risk' area)? The ratio is 1.0 for a perfectly uniform,
fully-deep survey and drops as more area is shallow/non-uniform.

In [ ]:
from rubin_sim.maf.metrics.cosmology_summary_metrics import UniformAreaFoMFractionMetric

# UniformAreaFoMFractionMetric: summary metric. Compares the per-pixel depth map (from the metric below) to
# the depth a perfectly uniform survey would deliver, and returns the FRACTION of the ideal DESC Figure of
# Merit (FoM) that survives given the actual (non-uniform) depth pattern -- i.e. how much cosmological
# constraining power is lost to non-uniformity/under-depth ('AreaAtRisk').
from rubin_sim.maf.metrics.uniformity_metrics import NestedRIZExptimeExgalM5Metric

# NestedRIZExptimeExgalM5Metric: per-pixel ('parent') metric. Computes the coadded r+i+z exposure time and
# the resulting extragalactic 5-sigma depth (ExgalM5, i.e. m5 corrected for Galactic dust extinction) reached
# in that pixel, applying a minimum-depth cut. This is the depth map fed into UniformAreaFoMFractionMetric.
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import (
    RIZDetectionCoaddExposureTime,
    ExgalM5WithCuts,
)
from rubin_sim.maf.metrics.exgal_m5 import ExgalM5

nside = 64

sim_list = [baseline_file]
name_list = [splitext(basename(sim))[0] for sim in sim_list]

years = range(1, 11)
surveyAreas = SkyAreaGenerator(nside=nside)
map_footprints, map_labels = surveyAreas.return_maps()
slicer = maf.HealpixSubsetSlicer(
    nside=nside,
    hpid=np.where(map_labels == "lowdust")[0],  # hpid=np.arange(hp.nside2npix(nside)),
    use_cache=False,
)

results_allruns = {}
for opsim_fname, run_name in zip(sim_list, name_list):
    results_allyears = np.zeros((len(years),))
    # loop over years
    for iy, year in enumerate(years):
        print("year", year)

        days = year * 365.25
        constraint_str = (
            'scheduler_note not like "DD%" and night <= XX and scheduler_note not like "twilight_near_sun" '
        )
        constraint_str = constraint_str.replace("XX", "%d" % days)

        # per-pixel coadded riz depth map, keeping only pixels reaching at least depth_cut
        metric = NestedRIZExptimeExgalM5Metric(
            depth_cut=25.0  # what depth cuts to apply year after year?
        )
        # reduce the depth map to a single 'FoMRatio' number: fraction of the ideal FoM retained given
        # the depth map's non-uniformity, for this survey year
        summary_metrics = [
            UniformAreaFoMFractionMetric(year, nside=nside, verbose=False, metric_name="FoMRatio")
        ]
        depth_map_bundles = [
            maf.MetricBundle(
                metric=metric,
                slicer=slicer,
                constraint=constraint_str,
                run_name=run_name,
                summary_metrics=summary_metrics,
            )
        ]
        bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
        bgroup = maf.MetricBundleGroup(bd, opsim_fname)
        bgroup.run_all()
        results_allyears[iy] = bd[list(bd.keys())[0]].summary_values["FoMRatio"]

    results_allruns[run_name] = results_allyears

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(7, 4), sharex=True)
axs = [axs]
colors = ["orange", "blue", "black", "red"]
for i, run_name in enumerate(results_allruns.keys()):
    axs[0].plot(years, results_allruns[run_name], label=run_name, marker="o", color=colors[i])
axs[0].legend()

axs[0].set_xlabel("Years")
axs[0].set_ylabel("FOM ratio")

# Demo of meanz metric

This demo asks: given non-uniform depth, how biased is the MEAN REDSHIFT (<z>) inferred for each
tomographic bin? A biased <z> per bin is a key systematic for weak-lensing cosmology, since the
lensing signal amplitude depends directly on the assumed source-galaxy redshift distribution.

In [ ]:
from rubin_sim.maf.metrics.cosmology_summary_metrics import MultibandMeanzBiasMetric

# MultibandMeanzBiasMetric: summary metric. Uses the MEANZ_TOMOGRAPHY_MODEL (DESC's model of how the mean
# redshift of each tomographic bin responds to per-band depth) together with the per-pixel multiband depth
# map below to estimate the resulting BIAS on mean redshift <z> for each bin, for the given survey year.
from rubin_sim.maf.metrics.uniformity_metrics import MultibandExgalM5

# MultibandExgalM5: per-pixel ('parent') metric. Simply computes the extragalactic 5-sigma depth
# (dust-corrected m5) in every LSST band for that pixel; this multiband depth map is what
# MultibandMeanzBiasMetric converts into a mean-redshift bias.
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import (
    RIZDetectionCoaddExposureTime,
    ExgalM5WithCuts,
)
from rubin_sim.maf.metrics.exgal_m5 import ExgalM5
from rubin_sim.maf.metrics.tomography_models import DENSITY_TOMOGRAPHY_MODEL, MEANZ_TOMOGRAPHY_MODEL

nside = 32

sim_list = [baseline_file]
name_list = [splitext(basename(sim))[0] for sim in sim_list]

years = range(1, 10)
surveyAreas = SkyAreaGenerator(nside=nside)
map_footprints, map_labels = surveyAreas.return_maps()
slicer = maf.HealpixSubsetSlicer(
    nside=nside,
    hpid=np.where(map_labels == "lowdust")[0],  # hpid=np.arange(hp.nside2npix(nside)),
    use_cache=False,
)

results_allruns = {}
for opsim_fname, run_name in zip(sim_list, name_list):
    results_allyears = np.zeros((len(years),))
    # loop over years
    for iy, year in enumerate(years):
        print("year", year)

        days = year * 365.25
        constraint_str = (
            'scheduler_note not like "DD%" and night <= XX and scheduler_note not like "twilight_near_sun" '
        )
        constraint_str = constraint_str.replace("XX", "%d" % days)

        # per-pixel multiband (ugrizy) extragalactic depth map
        metric = MultibandExgalM5()
        # reduce the multiband depth map to a bias on mean redshift <z>, per tomographic bin, for this year
        summary_metrics = [
            MultibandMeanzBiasMetric(MEANZ_TOMOGRAPHY_MODEL, year=year, metric_name="MultibandMeanzBias")
        ]
        depth_map_bundles = [
            maf.MetricBundle(
                metric=metric,
                slicer=slicer,
                constraint=constraint_str,
                run_name=run_name,
                summary_metrics=summary_metrics,
            )
        ]
        bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
        bgroup = maf.MetricBundleGroup(bd, opsim_fname)
        bgroup.run_all()
        # take the lowest z-bin I think
        # summary_values["MultibandMeanzBias"] is a list of dicts, one per tomographic bin; here we keep bin 0
        # (lowest-redshift bin) and its 'y1ratio' entry, a normalized measure of the meanz bias for that bin
        results_allyears[iy] = bd[list(bd.keys())[0]].summary_values["MultibandMeanzBias"][0]["y1ratio"]

    results_allruns[run_name] = results_allyears

In [ ]:
results_allruns